[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Concurrency and WAL


## What you will be able to do

Say what `database is locked` means and which connection caused it, from the locks SQLite takes on a
database file in its default rollback journal mode. Set how long a connection waits for a lock, and
recognize the deadlock that no wait can end, which `BEGIN IMMEDIATE` prevents. Switch a database to
write-ahead logging, explain what readers see while a writer commits, keep the `-wal` file from
growing, and name the limit WAL leaves in place: one writer at a time.


## The idea

### The problem

The stations database has more than one user now. A loader writes new readings every hour, a web
page shows the latest readings to whoever asks, and a report reads the whole year every night. Each
opens its own connection to `stations.db`, and most of the time they never notice each other.

Then the loader fails with `database is locked` while the report is running. The page, left open all
afternoon, still shows the readings from lunchtime. Two loaders counting the hours they have loaded
both add one, and the count goes up by one. The **Connections and Cursors** notebook met a cursor
that locked out a writer, and the **Transactions** notebook showed that a second connection does not
see rows until they are committed. Behind all of these are the locks SQLite takes on the file, and
which lock a connection holds depends on what it has done, not just on what it is doing now. Waiting
longer fixes some of these failures and does nothing for others, and knowing which is which is most
of the work.

### What locks and WAL are

> SQLite lets many connections read a database at once and one change it at a time, and keeps them
> apart with **locks** on the database file. In the default **rollback journal** mode, a connection
> that reads holds a **SHARED** lock, one that has begun to write holds a **RESERVED** lock beside
> the readers, and a commit needs an **EXCLUSIVE** lock, which waits until every reader has finished.
> When a lock is not free, SQLite retries for the connection's **busy timeout**, the `timeout` passed
> to `sqlite3.connect`, five seconds by default, and then fails with `database is locked`. In **WAL**
> mode, set with `PRAGMA journal_mode = WAL`, a commit appends the changed pages to a **write-ahead
> log**, the `-wal` file beside the database, so a reader keeps reading the version it started with
> while a writer commits, and a **checkpoint** later copies the log into the database file. WAL still
> allows one writer at a time.

### Why it works that way

- **One file, and no server.** Nothing coordinates the connections but the file, so a lock on the
  file decides who may change it, and SQLite gives that right to one connection at a time.
- **A lock is held until the transaction ends.** A reader's SHARED lock lasts until its transaction
  commits or rolls back, or until its query has been read to the end, so a reader that stops partway
  keeps a writer's commit waiting.
- **Waiting helps only when the lock will be released.** A connection that waits for a writer to
  commit succeeds once it does. Two transactions that have both read and both want to write would
  each wait for the other forever, so SQLite fails one of them at once, however long its timeout.
  `BEGIN IMMEDIATE` takes the write lock before the transaction reads anything, so a second writer
  waits at its `BEGIN`, where waiting works.
- **WAL gives every reader a snapshot.** A reader reads the database file together with the part of
  the log that was committed when its transaction began, so a commit neither waits for readers nor
  disturbs them.
- **A checkpoint cannot pass a reader.** Pages that a reader's snapshot still needs stay in the log,
  so a read transaction that never ends lets the `-wal` file keep growing. SQLite checkpoints on its
  own once the log reaches 1,000 pages.
- **WAL belongs to the file.** The setting stays with the database for every connection and program
  that opens it later, and because WAL's readers share memory through the `-shm` file, every
  connection has to be on the same computer.

### Where this shows up

A web application on SQLite meets these locks first. The **Django, Deep Dive** and **Flask, Deep
Dive** guides serve pages from several threads or processes at once, each with its own connection,
and `database is locked` in their logs means what it means here. PostgreSQL, in the **asyncpg and
psycopg3, Deep Dive** guide, lets many connections write at once, locking rows instead of the whole
database, which is the point at which a server earns its place. In this guide, the **Backup and
Copying** notebook copies a database in WAL mode, where the `-wal` file holds commits that the
database file does not have yet.

### What this notebook covers

- `database is locked`, from two connections that both want to write
- SHARED, RESERVED, PENDING and EXCLUSIVE locks in the rollback journal mode, and a cursor read
  partway
- The busy timeout, and a connection that waits for another thread's commit
- The deadlock that waiting cannot end, and `BEGIN IMMEDIATE`
- WAL: `journal_mode`, the `-wal` and `-shm` files, and readers during a commit
- Writers in WAL, and the snapshot a deferred transaction cannot write from
- Checkpoints, and `PRAGMA wal_checkpoint(TRUNCATE)`
- `synchronous = NORMAL`, and what a commit costs
- When to use the rollback journal, WAL, or a database server
- A loader and a page working at the same time, from two threads
- Seven errors: a report that locks out the loader, WAL set inside a transaction, a table dropped
  while it is read, a page that never sees new readings, a `-wal` file that keeps growing, a lost
  update, and WAL on a database in memory

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
from pathlib import Path

path = Path("first_look.db")
loader = sqlite3.connect(path, autocommit=True, timeout=0.1)
page = sqlite3.connect(path, autocommit=True, timeout=0.1)
loader.execute("CREATE TABLE readings (hour TEXT, celsius REAL)")

loader.execute("BEGIN IMMEDIATE")
loader.execute("INSERT INTO readings VALUES ('2026-01-01T00:00', -3.5)")
try:
    page.execute("INSERT INTO readings VALUES ('2026-01-01T00:00', -3.4)")
except sqlite3.OperationalError as error:
    print("page's insert:", error)
print("page reads:", page.execute("SELECT COUNT(*) FROM readings").fetchone()[0])

loader.execute("COMMIT")
count = page.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
print("after the loader's commit, page reads:", count)
loader.close()
page.close()
path.unlink()
```

```
page's insert: database is locked
page reads: 0
after the loader's commit, page reads: 1
```

Two connections to one file. While the loader's transaction was open, the page could read but not
write, and its insert failed after waiting its tenth of a second. The page read the table as it was
last committed, empty, until the loader's commit, and saw the new reading at once afterwards.


## Setup

Seven imports, and the stations and their year of readings in `stations.db`, with a copy,
`rollback.db`, that stays in the default rollback journal mode after `stations.db` changes to WAL.

- `sqlite3` builds the database, and every connection to it
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `threading` runs a second connection at the same time as the first, in the worked examples
- `time` measures how long a statement waited, and holds a lock for a moment
- `Path` names the scratch folder and the databases in it
- `shutil` copies the database, and removes the scratch folder at the end

`connect` opens a connection that writes its own `BEGIN` and `COMMIT`, with `autocommit=True`, and
waits a tenth of a second for a lock unless told otherwise. `attempt` runs one statement and says
what happened: its rows or its error, and whether that came at once or after waiting for a lock.


In [1]:
import math
import shutil
import sqlite3
import threading
import time
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
ROLLBACK = SCRATCH / "rollback.db"
for leftover in SCRATCH.glob("*.db*"):
    leftover.unlink()
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
NEW_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"
COUNT_2026 = "SELECT COUNT(*) FROM readings WHERE hour >= '2026'"


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE counters (name TEXT PRIMARY KEY, value INTEGER NOT NULL);
    INSERT INTO counters VALUES ('hours loaded', 8760);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany(NEW_READING, ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()
shutil.copy(DATABASE, ROLLBACK)


def connect(path=DATABASE, timeout=0.1):
    """A connection that writes its own BEGIN and COMMIT, and waits `timeout` seconds for a lock."""
    return sqlite3.connect(path, autocommit=True, timeout=timeout)


def attempt(conn, sql, parameters=()):
    """Run one statement, and say what happened: its rows or its error, at once or after waiting for a lock."""
    started = time.perf_counter()
    try:
        cursor = conn.execute(sql, parameters)
        outcome = f"ran {cursor.fetchall()}" if cursor.description else "ran"
    except sqlite3.OperationalError as error:
        outcome = f"OperationalError: {error}"
    return outcome + (", after waiting" if time.perf_counter() - started > 0.05 else ", at once")


print("built", DATABASE, "and", ROLLBACK)


built scratch/stations.db and scratch/rollback.db


## Worked examples

### Two writers

A loader and a page, each with its own connection. The loader begins a write and inserts a reading,
and the page tries to do the same before the loader commits:


In [2]:
loader, page = connect(), connect()

print("loader BEGIN IMMEDIATE:", attempt(loader, "BEGIN IMMEDIATE"))
print("loader INSERT:         ", attempt(loader, NEW_READING, (ids["Bergen"], "2026-01-01T00:00", 4.5)))
print("page INSERT:           ", attempt(page, NEW_READING, (ids["Oslo"], "2026-01-01T00:00", 1.5)))
print("page SELECT:           ", attempt(page, COUNT_2026))
print("loader COMMIT:         ", attempt(loader, "COMMIT"))
print("page INSERT, again:    ", attempt(page, NEW_READING, (ids["Oslo"], "2026-01-01T00:00", 1.5)))
print("page SELECT, again:    ", attempt(page, COUNT_2026))
loader.close()
page.close()


loader BEGIN IMMEDIATE: ran, at once
loader INSERT:          ran, at once
page INSERT:            OperationalError: database is locked, after waiting
page SELECT:            ran [(0,)], at once
loader COMMIT:          ran, at once
page INSERT, again:     ran, at once
page SELECT, again:     ran [(2,)], at once


While the loader's write was open, the page's insert waited its tenth of a second for the lock and
failed with `database is locked`. Reading was never refused: the page saw no readings from 2026,
since the loader had committed none. Once the loader committed, the page's insert ran at once, and
its query counted both readings. With `autocommit=True`, the page's `INSERT` was a transaction of
its own, and needed the same write lock as the loader's.

### What a reader holds

In the rollback journal mode, a transaction that reads keeps its SHARED lock until it ends, and a
commit needs every reader gone. `rollback.db` is still in that mode:


In [3]:
report, loader = connect(ROLLBACK), connect(ROLLBACK)

print("report BEGIN:          ", attempt(report, "BEGIN"))
print("report SELECT:         ", attempt(report, "SELECT COUNT(*) FROM readings"))
print("loader BEGIN:          ", attempt(loader, "BEGIN"))
print("loader INSERT:         ", attempt(loader, NEW_READING, (ids["Tromso"], "2026-01-01T00:00", -4.5)))
print("loader COMMIT:         ", attempt(loader, "COMMIT"))
print("report COMMIT:         ", attempt(report, "COMMIT"))
print("loader COMMIT, again:  ", attempt(loader, "COMMIT"))


report BEGIN:           ran, at once
report SELECT:          ran [(35040,)], at once
loader BEGIN:           ran, at once
loader INSERT:          ran, at once
loader COMMIT:          OperationalError: database is locked, after waiting
report COMMIT:          ran, at once
loader COMMIT, again:   ran, at once


The loader could begin its write and insert while the report read, and its commit waited and failed:
a commit rewrites the database file, which it may not do while anyone is reading it. The failed
commit left the loader's transaction open, so once the report ended its transaction, the same
`COMMIT` ran. SQLite's locks, from the weakest:

| Lock | Held by | Lets other connections |
|---|---|---|
| SHARED | a connection reading | read, and begin a write |
| RESERVED | a connection that has begun to write | keep reading, and begin reading |
| PENDING | a writer waiting to commit | finish reading, but not begin |
| EXCLUSIVE | a writer committing | do nothing |

A reader does not need a `BEGIN` to hold a lock. A query that has not been read to the end keeps its
SHARED lock too, as the **Connections and Cursors** notebook found:


In [4]:
cursor = report.execute("SELECT hour FROM readings")
print("report reads one row:", cursor.fetchone())
print("loader INSERT:       ", attempt(loader, NEW_READING, (ids["Tromso"], "2026-01-01T01:00", -4.9)))

cursor.close()
print("loader INSERT, after cursor.close():", attempt(loader, NEW_READING, (ids["Tromso"], "2026-01-01T01:00", -4.9)))
report.close()
loader.close()


report reads one row: ('2025-01-01T00:00',)
loader INSERT:        OperationalError: database is locked, after waiting
loader INSERT, after cursor.close(): ran, at once


The cursor had read one of 35,041 rows, and its query was still under way, holding its lock, so the
loader's insert, a transaction of its own, could not commit. Closing the cursor ended the query and
released the lock. Reading to the end with `fetchall` would have done the same.

### Waiting for a lock

A connection opened with `timeout=5.0` retries a lock for up to five seconds, which sqlite3 passes
to SQLite as its busy timeout. Here a thread holds the write lock for half a second, and the main
thread's insert waits for it:


In [5]:
locked = threading.Event()


def hold_the_write_lock(seconds):
    """In a thread of its own, begin a write, say so through `locked`, hold the lock, and commit."""
    holder = connect()
    holder.execute("BEGIN IMMEDIATE")
    holder.execute(NEW_READING, (ids["Svalbard"], "2026-01-01T00:00", -12.5))
    locked.set()
    time.sleep(seconds)
    holder.execute("COMMIT")
    holder.close()


patient = connect(timeout=5.0)
print("busy_timeout, in milliseconds:", patient.execute("PRAGMA busy_timeout").fetchone()[0])

thread = threading.Thread(target=hold_the_write_lock, args=(0.5,))
thread.start()
locked.wait()
started = time.perf_counter()
patient.execute(NEW_READING, (ids["Tromso"], "2026-01-01T00:00", -5.5))
waited = time.perf_counter() - started
thread.join()

print("the insert waited for the thread's commit, then ran:", 0.3 < waited < 5.0)
print("readings from 2026:", patient.execute(COUNT_2026).fetchone()[0])
patient.close()


busy_timeout, in milliseconds: 5000
the insert waited for the thread's commit, then ran: True
readings from 2026: 4


`locked` is a `threading.Event`, which the thread sets once it holds the lock, so the main thread's
insert is sure to arrive while the lock is held. The insert waited about half a second, until the
thread committed, and then ran. `PRAGMA busy_timeout` shows the same wait in milliseconds, and
setting it changes the wait on a connection that is already open.

### When waiting cannot help

Two transactions that both read before they write. Each connection waits up to five seconds:


In [6]:
first, second = connect(timeout=5.0), connect(timeout=5.0)
for conn in (first, second):
    conn.execute("BEGIN")
    conn.execute(COUNT_2026).fetchone()

print("first INSERT: ", attempt(first, NEW_READING, (ids["Bergen"], "2026-01-01T01:00", 4.1)))
print("second INSERT:", attempt(second, NEW_READING, (ids["Oslo"], "2026-01-01T01:00", 1.1)))
print("second ROLLBACK:", attempt(second, "ROLLBACK"))
print("first COMMIT: ", attempt(first, "COMMIT"))


first INSERT:  ran, at once
second INSERT: OperationalError: database is locked, at once
second ROLLBACK: ran, at once
first COMMIT:  ran, at once


The second insert failed at once, with five seconds of patience unused. Both transactions held SHARED
locks from their reads. The first wrote, taking the RESERVED lock, and could commit only once the
second stopped reading, while the second could write only once the first had committed. Neither could
ever go first, so SQLite refused the second at once instead of letting both wait out their timeouts.

`BEGIN IMMEDIATE` takes the RESERVED lock at the start, before the transaction reads anything. A
second writer then waits at its own `BEGIN IMMEDIATE`, holding nothing the first needs, and goes
ahead once the first commits. The same two transactions, written that way, with the first in a
thread:


In [7]:
writing = threading.Event()


def read_then_write(conn, station, hour, celsius, pause=0.0):
    """One transaction that takes the write lock, counts the readings from 2026, and inserts one."""
    conn.execute("BEGIN IMMEDIATE")
    writing.set()
    conn.execute(COUNT_2026).fetchone()
    time.sleep(pause)
    conn.execute(NEW_READING, (ids[station], hour, celsius))
    conn.execute("COMMIT")


def bergen_in_a_thread():
    """The first transaction, on a connection this thread opens and closes."""
    conn = connect(timeout=5.0)
    read_then_write(conn, "Bergen", "2026-01-01T02:00", 3.9, pause=0.3)
    conn.close()


thread = threading.Thread(target=bergen_in_a_thread)
thread.start()
writing.wait()
read_then_write(second, "Oslo", "2026-01-01T02:00", 0.8)
thread.join()

print("readings from 2026:", second.execute(COUNT_2026).fetchone()[0])
first.close()
second.close()


readings from 2026: 7


The thread's transaction took the write lock first, and the main thread's `BEGIN IMMEDIATE` waited
for its commit, then read, wrote and committed in turn. Both readings arrived. `writing` makes sure
the thread holds the lock before the main thread begins, and the thread opens and closes a
connection of its own, since a sqlite3 connection refuses to be used from a thread other than the
one that created it.

### WAL

`PRAGMA journal_mode = WAL` changes the file, and returns the mode in force. A new connection, and
every program that opens the file later, finds WAL already set. Then the report and the loader of
the earlier example, on `stations.db`:


In [8]:
switch = connect()
print("journal_mode:", switch.execute("PRAGMA journal_mode = WAL").fetchone()[0])
switch.close()

report, loader = connect(), connect()
print("a new connection's journal_mode:", report.execute("PRAGMA journal_mode").fetchone()[0])

print("report BEGIN:", attempt(report, "BEGIN"))
print("report reads:", attempt(report, COUNT_2026))
print("loader INSERT:", attempt(loader, NEW_READING, (ids["Svalbard"], "2026-01-01T01:00", -12.9)))
print("report reads, in the same transaction:", attempt(report, COUNT_2026))
print("report COMMIT:", attempt(report, "COMMIT"))
print("report reads, in a new transaction:", attempt(report, COUNT_2026))
print("files:", sorted(path.name for path in SCRATCH.iterdir()))


journal_mode: wal
a new connection's journal_mode: wal
report BEGIN: ran, at once
report reads: ran [(7,)], at once
loader INSERT: ran, at once
report reads, in the same transaction: ran [(7,)], at once
report COMMIT: ran, at once
report reads, in a new transaction: ran [(8,)], at once
files: ['rollback.db', 'stations.db', 'stations.db-shm', 'stations.db-wal']


The loader's insert committed at once while the report's transaction was open, which the rollback
journal mode refused. The report went on seeing the database as it was when its transaction began,
seven readings from 2026, and saw the eighth as soon as it began a new transaction. The loader's
commit went into `stations.db-wal`, and `stations.db-shm` is the shared memory the connections use
to find their way around the log. Both files belong to the database while it is open, and SQLite
removes them when the last connection closes.

### Writers in WAL

WAL separates readers from the writer, and still allows one writer at a time:


In [9]:
print("loader BEGIN IMMEDIATE:", attempt(loader, "BEGIN IMMEDIATE"))
print("report BEGIN IMMEDIATE:", attempt(report, "BEGIN IMMEDIATE"))
print("loader COMMIT:         ", attempt(loader, "COMMIT"))

report.execute("BEGIN")
report.execute(COUNT_2026).fetchone()
loader.execute(NEW_READING, (ids["Bergen"], "2026-01-01T03:00", 3.6))
try:
    report.execute(NEW_READING, (ids["Oslo"], "2026-01-01T03:00", 0.4))
except sqlite3.OperationalError as error:
    print("report INSERT, from an old snapshot:", error, error.sqlite_errorname)
print("report ROLLBACK:", attempt(report, "ROLLBACK"))


loader BEGIN IMMEDIATE: ran, at once
report BEGIN IMMEDIATE: OperationalError: database is locked, after waiting
loader COMMIT:          ran, at once
report INSERT, from an old snapshot: database is locked SQLITE_BUSY_SNAPSHOT
report ROLLBACK: ran, at once


A second `BEGIN IMMEDIATE` waited and failed while the first write was open, as in the rollback
journal mode. The last case fails at once, and for a different reason. The report's transaction read
a snapshot, then the loader committed a newer version of the database, and a transaction may not
write on top of a version it has not seen. `sqlite_errorname` tells the two failures apart:
`SQLITE_BUSY` for a lock that was taken, `SQLITE_BUSY_SNAPSHOT` for a snapshot that went out of
date. Waiting cannot help the second, and `BEGIN IMMEDIATE` prevents it again.

### Checkpoints

A checkpoint copies committed pages from the `-wal` file into the database file.
`PRAGMA wal_checkpoint(TRUNCATE)` does all of it and empties the log, and returns three numbers:
whether it was blocked, the pages in the log, and the pages copied. With a reader's transaction
open, and then without:


In [10]:
WAL_FILE = SCRATCH / "stations.db-wal"

report.execute("BEGIN")
report.execute(COUNT_2026).fetchone()
blocked, _, _ = loader.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone()
print("checkpoint blocked by the open reader:", blocked == 1)

report.execute("COMMIT")
print("checkpoint once the reader finished:", loader.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone())
print("bytes left in the -wal file:", WAL_FILE.stat().st_size)


checkpoint blocked by the open reader: True
checkpoint once the reader finished: (0, 0, 0)
bytes left in the -wal file: 0


With the reader's transaction open, the checkpoint waited out the loader's timeout and reported
itself blocked, since the reader's snapshot still needed pages from the log. Once the reader
finished, `(0, 0, 0)` means that nothing blocked the checkpoint and the log is empty, and `TRUNCATE`
cut the file to nothing. SQLite runs a smaller checkpoint by itself whenever the log passes 1,000
pages, so a program only calls this to keep the file small at a moment it chooses.

### synchronous = NORMAL

`PRAGMA synchronous` says how often SQLite waits for the operating system to confirm that data is on
the disk. `FULL`, the default, waits at every commit. In WAL mode, `NORMAL` waits at checkpoints
instead. Here 300 inserts, each committed on its own, into a table in the rollback journal mode with
`FULL`, and in WAL mode with `NORMAL`:


In [11]:
def time_commits(mode, synchronous):
    """Seconds taken by 300 inserts, each committed on its own, into a new database in this mode."""
    path = SCRATCH / f"timing_{mode}.db"
    timing = connect(path)
    timing.execute(f"PRAGMA journal_mode = {mode}")
    timing.execute(f"PRAGMA synchronous = {synchronous}")
    timing.execute("CREATE TABLE readings (hour TEXT, celsius REAL)")
    started = time.perf_counter()
    for n in range(300):
        timing.execute("INSERT INTO readings VALUES (?, ?)", (f"2026-01-01T{n % 24:02d}:00", 0.0))
    seconds = time.perf_counter() - started
    timing.close()
    return seconds


rollback_full = time_commits("DELETE", "FULL")
wal_normal = time_commits("WAL", "NORMAL")
print("the rollback journal with FULL took more than 5 times as long:", rollback_full > 5 * wal_normal)
print("synchronous on a new connection:", loader.execute("PRAGMA synchronous").fetchone()[0])


the rollback journal with FULL took more than 5 times as long: True
synchronous on a new connection: 2


On the machine this notebook was written on, the commits in WAL mode with `NORMAL` were dozens of
times faster. The price is written in SQLite's documentation: a database in WAL mode with `NORMAL`
cannot be corrupted by a crash or a power cut, but the last transactions before a power cut can be
lost. `synchronous` belongs to the connection, not the file, so `loader` still reports 2, `FULL`,
and every connection that wants `NORMAL` sets it. The mode and setting in the f-string come from the
code, never from input.

### Rollback journal, WAL, or a server

| Write | When | Why |
|---|---|---|
| the default rollback journal | one program at a time, or a database copied about as a single file | nothing to set up, and the database is always one complete file |
| `PRAGMA journal_mode = WAL` | a reader and a writer at the same time, such as a page and a loader, on one computer | readers do not wait for the writer, nor the writer for readers |
| `BEGIN IMMEDIATE` | any transaction that reads and then writes, with other writers about | a second writer waits at its `BEGIN`, where waiting works, instead of failing at its first write |
| a database server, such as PostgreSQL | many writers at once, or connections from other computers | SQLite allows one writer at a time, and WAL needs every connection on one computer |

The default for a database that several connections use is WAL, a `timeout` of a few seconds, short
write transactions, and `BEGIN IMMEDIATE` for every one that reads first.

### A loader and a page, at the same time

The pieces of this notebook together: in WAL mode, a loader thread writes 48 hourly readings, each in
its own short `BEGIN IMMEDIATE` transaction, while a page thread reads the latest count again and
again. Both wait up to five seconds for a lock, and both write down any error:


In [12]:
errors, counts = [], []
loading = threading.Event()


def load_hours():
    """Write 48 hours of readings for Kirkenes, one short transaction each."""
    conn = connect(timeout=5.0)
    loading.set()
    try:
        for hour in range(48):
            conn.execute("BEGIN IMMEDIATE")
            day_and_hour = f"2026-01-{1 + hour // 24:02d}T{hour % 24:02d}:00"
            conn.execute(NEW_READING, (ids["Kirkenes"], day_and_hour, -7.0))
            conn.execute("UPDATE counters SET value = value + 1 WHERE name = 'hours loaded'")
            conn.execute("COMMIT")
            time.sleep(0.005)
    except sqlite3.Error as error:
        errors.append(error)
    finally:
        loading.clear()
        conn.close()


def show_page():
    """Read Kirkenes's latest count while the loader runs."""
    conn = connect(timeout=5.0)
    try:
        while loading.is_set():
            count = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ?", (ids["Kirkenes"],)).fetchone()[0]
            counts.append(count)
            time.sleep(0.002)
    except sqlite3.Error as error:
        errors.append(error)
    finally:
        conn.close()


loader_thread = threading.Thread(target=load_hours)
loader_thread.start()
loading.wait()
page_thread = threading.Thread(target=show_page)
page_thread.start()
loader_thread.join()
page_thread.join()

print("errors:", errors)
kirkenes = loader.execute("SELECT COUNT(*) FROM readings WHERE station_id = ?", (ids["Kirkenes"],)).fetchone()[0]
print("Kirkenes readings now:", kirkenes)
print("hours loaded:", loader.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0])
print("the page saw the count grow, and never go down:", len(set(counts)) > 1 and counts == sorted(counts))
loader.close()
report.close()


errors: []
Kirkenes readings now: 48
hours loaded: 8808
the page saw the count grow, and never go down: True


No errors from either thread. The loader's 48 commits went through while the page read, and the
page's reads went on while the loader wrote. Every count the page saw was a committed state, never
half a transaction, so the counts only rose. The counter rose with the readings, 48 hours on top of
8,760, because every increment was `value = value + 1` inside the same transaction as its reading,
as the lost update in Common errors explains.

### Where each part came from

| In the loader and the page | What it relies on | The section that showed it |
|---|---|---|
| `stations.db` in WAL mode | readers and a writer that do not wait for each other | WAL |
| `connect(timeout=5.0)` in both threads | a connection that waits for a lock it can get | Waiting for a lock |
| `BEGIN IMMEDIATE` around each reading | the write lock taken before anything is read | When waiting cannot help |
| one short transaction for every hour | a lock held for a moment, not for the whole load | Two writers |
| counts that only rise | every read sees a committed snapshot | WAL |
| a connection opened inside each thread | a connection used only by the thread that made it | When waiting cannot help |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/17-concurrency-and-wal-solutions.ipynb).

**1.** On `rollback.db`, begin a write on one connection and insert a reading, then show whether a
second connection can read while that transaction is open, and whether it sees the new reading.


In [13]:
# your code here


**2.** On a connection opened with `timeout=0.1`, set `PRAGMA busy_timeout` to 400, and show that an
insert blocked by another connection's open write now waits longer before it fails.


In [14]:
# your code here


**3.** In WAL mode, begin a transaction on one connection and read the count of readings from 2026,
insert three readings from another connection, and print what the first connection sees before and
after it ends its transaction.


In [15]:
# your code here


**4.** Show that `PRAGMA journal_mode` is a property of the file: set WAL on a new database, close
every connection, open a new one, and print its journal mode.


In [16]:
# your code here


**5.** Two connections each read the counter `'hours loaded'` and write back the value plus one. Make
the increments safe with `BEGIN IMMEDIATE` around each read and write, and show the counter rose by
two.


In [17]:
# your code here


**6.** Set a database back to `DELETE` journal mode, close every connection, and show that the `-wal`
and `-shm` files are gone.


In [18]:
# your code here


## Common errors

### sqlite3.OperationalError: database is locked


In [19]:
report = sqlite3.connect(ROLLBACK, autocommit=False, timeout=0.1)
print("report reads:", report.execute("SELECT COUNT(*) FROM readings").fetchone()[0])

loader = connect(ROLLBACK)
loader.execute(NEW_READING, (ids["Bergen"], "2026-01-02T00:00", 4.2))


report reads: 35042


OperationalError: database is locked

The report opened its connection with `autocommit=False`, which keeps a transaction open at all
times, as the **autocommit and isolation_level** notebook showed, and its query took a SHARED lock
that the transaction keeps. The report never committed, so in the rollback journal mode the loader's
insert could not commit, however long it waited. End the reader's transaction when it has read what
it needs:


In [20]:
report.commit()
print("loader INSERT:", attempt(loader, NEW_READING, (ids["Bergen"], "2026-01-02T00:00", 4.2)))
report.close()
loader.close()


loader INSERT: ran, at once


In WAL mode the same insert would have run, since a reader's snapshot does not block a commit, though
the report would then have gone on seeing the database as it was when its transaction began.

### sqlite3.OperationalError: cannot change into wal mode from within a transaction


In [21]:
converter = sqlite3.connect(ROLLBACK, autocommit=False)
converter.execute("PRAGMA journal_mode = WAL")


OperationalError: cannot change into wal mode from within a transaction

A connection with `autocommit=False` begins a transaction as it connects, and the journal mode can
change only between transactions. Change it on a connection that is outside a transaction, such as
one with `autocommit=True`, before it begins any:


In [22]:
converter.close()
converter = connect(ROLLBACK)
print("journal_mode:", converter.execute("PRAGMA journal_mode = WAL").fetchone()[0])
print("and back:", converter.execute("PRAGMA journal_mode = DELETE").fetchone()[0])
converter.close()


journal_mode: wal
and back: delete


### sqlite3.OperationalError: database table is locked


In [23]:
cleaner = connect(ROLLBACK)
for (name,) in cleaner.execute("SELECT name FROM sqlite_schema WHERE type = 'table'"):
    cleaner.execute(f'DROP TABLE "{name}"')


OperationalError: database table is locked

This lock is inside one connection. The loop was still reading the rows of its query about
`sqlite_schema` when the first `DROP TABLE` tried to change that same schema, and SQLite refuses to
drop a table while a statement on the same connection is using it. The message says
`database table`, and the error's name is `SQLITE_LOCKED`, where two connections give
`database is locked` and `SQLITE_BUSY`. Read everything first, then change it:


In [24]:
tables = cleaner.execute("SELECT name FROM sqlite_schema WHERE type = 'table'").fetchall()
for (name,) in tables:
    cleaner.execute(f'DROP TABLE "{name}"')
print("tables left:", cleaner.execute("SELECT COUNT(*) FROM sqlite_schema WHERE type = 'table'").fetchone()[0])
cleaner.close()


tables left: 0


The table names come from `sqlite_schema`, never from input.

### No error, and a page that never sees new readings: a read transaction left open


In [25]:
page = sqlite3.connect(DATABASE, autocommit=False)
print("page counts readings from 2026:", page.execute(COUNT_2026).fetchone()[0])

loader = connect()
loader.execute(NEW_READING, (ids["Oslo"], "2026-01-02T00:00", 0.9))
print("loader counts:", loader.execute(COUNT_2026).fetchone()[0])
print("page counts again:", page.execute(COUNT_2026).fetchone()[0])


page counts readings from 2026: 57
loader counts: 58
page counts again: 57


The loader's reading committed, and the page did not see it. In WAL mode nothing waited and nothing
failed, but the page's connection, with `autocommit=False`, was still inside the transaction its
first query began, reading the snapshot from that moment, and would go on reading it for as long as
the page stayed open. A page that reads should end its transaction after every request:


In [26]:
page.commit()
print("page counts, after commit():", page.execute(COUNT_2026).fetchone()[0])
page.commit()


page counts, after commit(): 58


### No error, and a -wal file that keeps growing: a reader that never finishes


In [27]:
def wal_bytes_after_commits(reader_open):
    """The size of the -wal file after 3,000 separately committed inserts, with or without a reader's transaction open."""
    reader = connect()
    if reader_open:
        reader.execute("BEGIN")
        reader.execute(COUNT_2026).fetchone()
    for n in range(3000):
        loader.execute("INSERT INTO counters VALUES (?, ?)", (f"test {n}", n))
    size = WAL_FILE.stat().st_size
    if reader_open:
        reader.execute("COMMIT")
    loader.execute("DELETE FROM counters WHERE name LIKE 'test %'")
    loader.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    reader.close()
    return size


print("the -wal file grew more than twice as large with a reader open:",
      wal_bytes_after_commits(reader_open=True) > 2 * wal_bytes_after_commits(reader_open=False))


the -wal file grew more than twice as large with a reader open: True


Without a reader, SQLite checkpointed every 1,000 pages or so and started the log again from its
beginning, so the file stopped growing. With a reader's transaction open, no checkpoint could pass
the pages its snapshot needed, and the log grew by every commit. A reader that stays in a
transaction for hours, such as the page above, can grow the file without limit. End read
transactions promptly, and check for a stuck reader when `PRAGMA wal_checkpoint` keeps reporting
itself blocked.

### No error, and a lost update: a count read and written in two steps


In [28]:
one, two = connect(), connect()
before = one.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]

seen_by_one = one.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
seen_by_two = two.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
one.execute("UPDATE counters SET value = ? WHERE name = 'hours loaded'", (seen_by_one + 1,))
two.execute("UPDATE counters SET value = ? WHERE name = 'hours loaded'", (seen_by_two + 1,))

after = one.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
print("two increments, and the counter rose by:", after - before)


two increments, and the counter rose by: 1


Both connections read the same value, both wrote that value plus one, and the second write replaced
the first: one of the increments was lost, with no lock broken and no error, since each statement
was a correct transaction by itself. Let SQL do the arithmetic inside one statement, which reads and
writes under one lock:


In [29]:
before = one.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
one.execute("UPDATE counters SET value = value + 1 WHERE name = 'hours loaded'")
two.execute("UPDATE counters SET value = value + 1 WHERE name = 'hours loaded'")
after = one.execute("SELECT value FROM counters WHERE name = 'hours loaded'").fetchone()[0]
print("two increments, and the counter rose by:", after - before)
one.close()
two.close()


two increments, and the counter rose by: 2


When the new value needs Python between the read and the write, put both inside one
`BEGIN IMMEDIATE` transaction, so the second connection waits to read until the first has written.

### No error, and no WAL: journal_mode on a database in memory


In [30]:
memory = sqlite3.connect(":memory:")
print("PRAGMA journal_mode = WAL returned:", memory.execute("PRAGMA journal_mode = WAL").fetchone()[0])
memory.close()


PRAGMA journal_mode = WAL returned: memory


The `PRAGMA` raised nothing and set nothing: a database in memory has no file to keep a log beside,
so it stays in the `memory` journal mode, and the `PRAGMA` returns the mode in force, not the one
asked for. A test that runs against `:memory:` never exercises WAL, whatever the code sets, so check
what the `PRAGMA` returned, and test concurrency against a file:


In [31]:
page.close()
loader.close()
report.close()
sidecars = sorted(path.name for path in SCRATCH.glob("stations.db-*"))
print("the -wal and -shm files, once every connection is closed:", sidecars)


the -wal and -shm files, once every connection is closed: []


Last, every connection is closed, so this cell removes the scratch folder, with the databases in it:


In [32]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- SQLite allows many readers and one writer at a time, and a connection that cannot get a lock waits
  for its `timeout`, five seconds by default, then fails with `database is locked`.
- In the rollback journal mode, a reader's transaction, or a query not read to the end, holds a
  SHARED lock, and a commit waits until every reader has finished.
- Two transactions that both read and then write can deadlock, and SQLite fails one at once, so begin
  every transaction that reads and then writes with `BEGIN IMMEDIATE`.
- `PRAGMA journal_mode = WAL` is set once, on the file, outside a transaction, and lets readers read
  a snapshot while a writer commits, with the commits kept in the `-wal` file until a checkpoint.
- A read transaction left open sees only its snapshot and holds back checkpoints, so end read
  transactions promptly.
- In WAL mode, `synchronous = NORMAL` makes commits much faster, at the risk of losing the last
  commits, never the database, in a power cut.
- Read-then-write in two statements loses updates, so increment in SQL, or inside `BEGIN IMMEDIATE`.


## What is next

The **Backup and Copying** notebook copies a database that is in use: why copying the file, or a WAL
database without its `-wal` file, is not a backup, and `conn.backup`, `VACUUM INTO` and `iterdump`,
which are.


---

&#8592; **Previous:** [Full-Text Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/16-full-text-search.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
